In [ ]:
# install required dependencies
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  !pip install -q pyomo
  !apt-get install -y -qq glpk-utils
  !apt-get install -y -qq coinor-cbc
  !wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"
  !unzip -o -q ipopt-linux64
  !pip install polytope

In [ ]:
import polytope as pt
import numpy as np
import matplotlib.pyplot as plt

# Helper Functions:

def minkowski_sum(X, Y):

    # Minkowski sum between two polytopes based on
    # vertex enumeration
    V_sum = []
    if isinstance(X, pt.Polytope):
        V1 = pt.extreme(X)
    else:
        # assuming vertices are in (N x d) shape. N # of vertices, d dimension
        V1 = X

    if isinstance(Y, pt.Polytope):
        V2 = pt.extreme(Y)
    else:
        V2 = Y

    for i in range(V1.shape[0]):
        for j in range(V2.shape[0]):
            V_sum.append(V1[i,:] + V2[j,:])
    return pt.qhull(np.asarray(V_sum))


def precursor(Xset, A, Uset=pt.Polytope(), B=np.array([])):
        if not B.any():
            return pt.Polytope(Xset.A @ A, Xset.b)
        else:
            tmp  = minkowski_sum( Xset, pt.extreme(Uset) @ -B.T )
        return pt.Polytope(tmp.A @ A, tmp.b)


def Oinf(Xset, A):
    Omega = Xset
    k = 0
    Omegap = precursor(Omega, A).intersect(Omega)
    while not Omegap == Omega:
        k += 1
        Omega = Omegap
        Omegap = pt.reduce(precursor(Omega, A).intersect(Omega))
    return Omegap


In [ ]:
from __future__ import division
import pyomo.environ as pyo
import numpy as np

def solve_cftoc(A, B, P, Q, R, N, x0, xL, xU, uL, uU, bf, Af):
  model = pyo.ConcreteModel()

  # Index sets.
  nx = np.size(A, 0)  # number of states
  nu = np.size(B, 1)  # number of inputs
  nf = np.size(Af, 0)
  model.tidx  = pyo.Set(initialize=range(N+1))
  model.xidx  = pyo.Set(initialize=range(nx))
  model.uidx  = pyo.Set(initialize=range(nu))
  model.nfidx = pyo.Set(initialize=range(nf))

  # State and input trajectory variables.
  model.x = pyo.Var(
      model.xidx, model.tidx,
      bounds=(xL,xU)
  )
  model.u = pyo.Var(
      model.uidx, model.tidx,
      bounds=(uL,uU)
  )

  # Initial state parameter.
  model.x0 = pyo.Param(model.xidx, initialize=x0, within=pyo.Any, mutable=True)

  # Objective.
  def objective_rule(m):
      costX = sum(
          m.x[i, t] * Q[i, j] * m.x[j, t]
          for t in m.tidx for i in m.xidx for j in m.xidx if t < N
      )
      costU = sum(
          m.u[i, t] * R[i, j] * m.u[j, t]
          for t in m.tidx for i in m.uidx for j in m.uidx if t < N
      )
      costTerminal = sum(
          m.x[i, N] * P[i, j] * m.x[j, N]
          for i in m.xidx for j in m.xidx
      )
      return costX + costU + costTerminal

  model.cost = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

  ## Constraints.
  # Dynamics constraint.
  def dynamics_rule(m, i, t):
      return m.x[i, t+1] == sum(A[i, j] * m.x[j, t] for j in m.xidx) \
                            + sum(B[i, j] * m.u[j, t] for j in m.uidx) \
              if t < N else pyo.Constraint.Skip
  model.dynamics_cons = pyo.Constraint(
      model.xidx, model.tidx, rule=dynamics_rule
  )

  # Initial state constraint.
  model.init_cons = pyo.Constraint(
      model.xidx, rule=lambda m, i: m.x[i, 0] == m.x0[i]
  )

  # Terminal constraint.
  model.terminal_cons = pyo.Constraint(
      model.nfidx,
      rule=lambda m, i: sum(Af[i, j] * m.x[j, N] for j in m.xidx) <= bf[i]
  )

  solver = pyo.SolverFactory('ipopt')

  # Set the initial state parameter.
  for i in model.xidx:
      model.x0[i].set_value(x0[i])

  # Solve the MPC problem.
  results = solver.solve(model)

  # Extract results.
  feas = str(results.solver.termination_condition) == "optimal"

  xOpt = np.asarray(
      [[pyo.value(model.x[i,t]) for i in model.xidx] for t in model.tidx]
  ).T
  # Corrected uOpt extraction to be nu x N matrix
  uOpt = np.asarray([[pyo.value(model.u[i,t]) for t in range(N)] for i in model.uidx])
  JOpt = model.cost()

  return [feas, xOpt, uOpt, JOpt]


In [ ]:
import numpy as np # Import numpy for math functions

def kinematic_truck_trailer(x, u, L=2.0, d=5.0):
    """
    Updated 6-state, 2-input kinematic model based on provided linearization.
    State: x = [x0_pos, y0_pos, theta0_truck_heading, theta1_trailer_articulation, v0_truck_velocity, phi0_truck_steering_angle]
    Control: u = [a_truck_acceleration, phidot_truck_steering_rate_change]
    """
    x0_pos, y0_pos, theta0_truck_heading, theta1_trailer_articulation, v0_truck_velocity, phi0_truck_steering_angle = \
        x[0], x[1], x[2], x[3], x[4], x[5]

    a_truck_acceleration, phidot_truck_steering_rate_change = u[0], u[1]

    # Model Equations
    dx0 = v0_truck_velocity * np.cos(theta0_truck_heading)
    dy0 = v0_truck_velocity * np.sin(theta0_truck_heading)
    dtheta0 = (v0_truck_velocity / L) * np.tan(phi0_truck_steering_angle)
    dtheta1 = (v0_truck_velocity / d) * np.sin(theta0_truck_heading - theta1_trailer_articulation)
    dv0 = a_truck_acceleration
    dphi0 = phidot_truck_steering_rate_change

    # System RHS: [dx0, dy0, dtheta0, dtheta1, dv0, dphi0]
    xdot = np.array([dx0, dy0, dtheta0, dtheta1, dv0, dphi0])

    return xdot

In [ ]:
import sympy
import numpy as np

def linearize_kinematic_truck_trailer(x_k, u_k, L=2.0, d=5.0):

    # 1. Define symbolic variables
    x0_s, y0_s, theta0_s, theta1_s, v0_s, phi0_s = sympy.symbols('x0 y0 theta0 theta1 v0 phi0')
    a_s, phidot_s = sympy.symbols('a phidot')
    L_s, d_s = sympy.symbols('L d')

    # 2. Reimplement the kinematic_truck_trailer model equations symbolically
    dx0_s = v0_s * sympy.cos(theta0_s)
    dy0_s = v0_s * sympy.sin(theta0_s)
    dtheta0_s = (v0_s / L_s) * sympy.tan(phi0_s)
    dtheta1_s = (v0_s / d_s) * sympy.sin(theta0_s - theta1_s) # Changed np.sin to sympy.sin
    dv0_s = a_s
    dphi0_s = phidot_s

    xdot_sym = sympy.Matrix([dx0_s, dy0_s, dtheta0_s, dtheta1_s, dv0_s, dphi0_s])

    # 3. Create symbolic vectors for state and control
    x_sym = sympy.Matrix([x0_s, y0_s, theta0_s, theta1_s, v0_s, phi0_s])
    u_sym = sympy.Matrix([a_s, phidot_s])

    # 4. Compute symbolic Jacobian matrices A and B
    A_sym = xdot_sym.jacobian(x_sym)
    B_sym = xdot_sym.jacobian(u_sym)

    # 5. Convert symbolic matrices to numerical functions using lambdify
    # Flatten the arguments for lambdify
    args = (x0_s, y0_s, theta0_s, theta1_s, v0_s, phi0_s, a_s, phidot_s, L_s, d_s) # Added phi0_s

    A_func = sympy.lambdify(args, A_sym, 'numpy')
    B_func = sympy.lambdify(args, B_sym, 'numpy')

    # 6. Evaluate the numerical A and B matrices at the given operating point
    # Unpack x_k and u_k for the lambdified functions
    x0, y0, theta0, theta1, v, phi = x_k[0], x_k[1], x_k[2], x_k[3], x_k[4], x_k[5]
    a, phi_dot = u_k[0], u_k[1]

    A_num = A_func(x0, y0, theta0, theta1, v, phi, a, phi_dot, L, d) # Added phi
    B_num = B_func(x0, y0, theta0, theta1, v, phi, a, phi_dot, L, d) # Added phi

    return A_num, B_num

In [ ]:
# Unit test.
A, B = linearize_kinematic_truck_trailer(x0, np.array([0.1, 0.05]))
nx_unit = A.shape[0]
nu_unit = B.shape[1]

# Fix: Q and P should be nx by nx, where nx is A.shape[0]
Q = np.eye(nx_unit)
# Fix: R should be nu by nu, where nu is B.shape[1]
R = np.eye(nu_unit)
P = Q
N = 15
xL = -15
xU = 15
uL = -1
uU = 1
x0 = np.array([0.0, 0.0, np.pi/4, np.pi/8, 1.0, np.pi/10]) # Adjusted x0 to match 6 states, initial state used for linearization

# Af and bf define the terminal constraint Af * x_N <= bf
# Example: constraining states to be within bounds for all 6 states
Af = np.array([[1, 0, 0, 0, 0, 0],
               [-1, 0, 0, 0, 0, 0],
               [0, 1, 0, 0, 0, 0],
               [0, -1, 0, 0, 0, 0],
               [0, 0, 1, 0, 0, 0],
               [0, 0, -1, 0, 0, 0],
               [0, 0, 0, 1, 0, 0],
               [0, 0, 0, -1, 0, 0],
               [0, 0, 0, 0, 1, 0],
               [0, 0, 0, 0, -1, 0],
               [0, 0, 0, 0, 0, 1],
               [0, 0, 0, 0, 0, -1]])
bf = np.array([5.0, 5.0, 5.0, 5.0, 1.0, 1.0, 0.5, 0.5, 0.1, 0.1, 2.0, 2.0]) # Relaxed terminal constraints for 6 states



[feas, xOpt, uOpt, JOpt] = solve_cftoc(A, B, P, Q, R, N, x0, xL, xU, uL, uU, bf, Af)

print('feas =', feas)
print('JOpt =', JOpt)
print('xOpt =\n', xOpt)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(xOpt.T)
ax[0].set_ylabel('x')
ax[1].plot(uOpt.T)
ax[1].set_ylabel('u')
plt.show()

In [ ]:
import numpy as np

# Fixed constants.

nx = 6 # Number of states for the truck-trailer model
nu = 2 # Number of inputs for the truck-trailer model
Q = np.eye(nx) # Q should be nx by nx
R = np.eye(nu) # R should be nu by nu
N = 3  # MPC horizon
xL = -15
xU = 15
uL = -1
uU = 1

M = 25  # simulation length
dt = 0.1 # Discretization time step

# Write your code here:
def simulate_mpc(P, bf, Af, x0, M):
  feas = np.zeros(M, dtype=bool)
  xOpt = np.zeros((nx, M+1))
  uOpt = np.zeros((nu, M))
  JOpt = np.zeros(M)
  xPred = np.zeros((nx, N+1, M))
  uPred = np.zeros((nu, N, M))
  predErr = np.zeros(M-N+1)
  x_current = np.copy(x0).astype(float) # Ensure x_current is float for operations

  # Initialize u_operating_point for the first linearization (e.g., zero control input)
  u_operating_point = np.array([0.5, 0.]) # Initial control input for linearization: v=0.5, delta=0

  for t in range(M):
    print(f"\n--- Simulation Step {t} ---")
    print(f"x_current (before MPC): {x_current}")
    print(f"u_operating_point for linearization: {u_operating_point}")

    # Dynamically linearize the continuous-time model at the current state and operating input
    A_c, B_c = linearize_kinematic_truck_trailer(x_current, u_operating_point)

    # Discretize the continuous-time A_c and B_c for the MPC solver
    A_d = np.eye(nx) + A_c * dt
    B_d = B_c * dt

    # Pass the dynamically generated A_d and B_d to solve_cftoc
    [feas_t, xPred_t, uPred_t, JOpt_t] = solve_cftoc(A_d, B_d, P, Q, R, N, x_current, xL, xU, uL, uU, bf, Af)
    feas[t] = feas_t
    JOpt[t] = JOpt_t

    if feas[t]:
      xOpt[:,t] = x_current # Store the state at the beginning of the step
      uOpt[:,t] = uPred_t[:,0] # Apply the first input from the optimal prediction

      # Update u_operating_point for the next linearization. It's the applied control.
      u_operating_point = uPred_t[:,0]

      print(f"uPred_t[:,0] (applied control): {uPred_t[:,0]}")
      nonlinear_xdot = kinematic_truck_trailer(x_current, uPred_t[:,0])
      print(f"kinematic_truck_trailer(x_current, uPred_t[:,0]) * dt: {nonlinear_xdot * dt}")

      # Update state using original nonlinear system dynamics
      x_current = x_current + nonlinear_xdot * dt
      print(f"x_current (after update): {x_current}")

      xPred[:,:,t] = xPred_t
      uPred[:,:,t] = uPred_t
    else:
      print(f"Infeasible at step {t}. Stopping simulation.")
      # If infeasible, set remaining feasibility to False and break
      feas[t:] = False
      # Fill remaining trajectory and predictions with NaN
      xOpt[:, t+1:] = np.nan
      uOpt[:, t:] = np.nan
      xPred[:, :, t:] = np.nan
      break # Stop simulation if infeasible

  # Store the last state if loop completes without infeasibility
  if np.all(feas): # Check if all previous steps were feasible
    xOpt[:,M] = x_current

  # Calculate predErr
  # predErr is a 1 x (M-N+1) array of the 2-norm of the difference
  # between the open-loop predictions and the closed-loop trajectory
  # for each state from simulation timestep 0 to M-N.
  for t in range(M - N + 1):
    if np.any(np.isnan(xOpt[:, t : t+N+1])) or np.any(np.isnan(xPred[:, :, t])):
        predErr[t] = np.nan # Indicate no valid comparison if data is missing
    else:
        predErr[t] = np.linalg.norm(xOpt[:, t : t+N+1] - xPred[:, :, t])

  # If any feasibility is False, return empty arrays as specified
  if not np.all(feas):
      xOpt = np.array([])
      uOpt = np.array([])
      xPred = np.array([])
      predErr = np.array([])

  return [feas, xOpt, uOpt, xPred, predErr]

In [ ]:
# Set terminal cost and constraints.
# Define nx and nu consistently for the truck-trailer model.
nx = 6 # Number of states
nu = 2 # Number of inputs

P = np.eye(nx) # Terminal cost P should be nx by nx

# Ensure Q and R are also defined for the correct dimensions here
Q = np.eye(nx) # Q should be nx by nx
R = np.eye(nu) # R should be nu by nu

# Af and bf define the terminal constraint Af * x_N <= bf
# Example: constraining states to be within +/- 0.1 of origin for all 4 states
Af = np.array([[1, 0, 0, 0, 0, 0],
               [-1, 0, 0, 0, 0, 0],
               [0, 1, 0, 0, 0, 0],
               [0, -1, 0, 0, 0, 0],
               [0, 0, 1, 0, 0, 0],
               [0, 0, -1, 0, 0, 0],
               [0, 0, 0, 1, 0, 0],
               [0, 0, 0, -1, 0, 0],
               [0, 0, 0, 0, 1, 0],
               [0, 0, 0, 0, -1, 0],
               [0, 0, 0, 0, 0, 1],
               [0, 0, 0, 0, 0, -1]])  # -x3 <= 1.0 (i.e., x3 >= -1.0)
bf = np.array([5.0, 5.0, 5.0, 5.0, 1.0, 1.0, 0.5, 0.5, 0.1, 0.1, 2.0, 2.0]) # Relaxed terminal constraints

x0 = np.array([5, 5, 0.5, -0.3, 0, 0])  # initial state, must be nx-dimensional, changed to a further point

# Unit test.
def plot_mpc_solution(feas, xOpt, uOpt, xPred, predErr):
    print('MPC Feasibility =\n', feas)
    if xOpt.size > 0: # Check if xOpt is not empty
        print('Optimal closed-loop trajectory =\n', xOpt)
    if uOpt.size > 0: # Check if uOpt is not empty
        print('Optimal inputs =\n', uOpt)
    print('Prediction error =\n', predErr)

    fig = plt.figure(figsize=(12, 8)) # Larger figure for 4 states
    # Plot states
    ax_x = fig.add_subplot(2, 1, 1)
    ax_x.plot(xOpt.T)
    ax_x.set_ylabel('States')
    ax_x.set_title('Closed-loop State Trajectories')
    ax_x.legend([f'x{i}' for i in range(nx)], loc='upper right')
    ax_x.grid(True)

    # Plot inputs
    ax_u = fig.add_subplot(2, 1, 2)
    ax_u.plot(uOpt.T)
    ax_u.set_ylabel('Inputs')
    ax_u.set_title('Optimal Control Inputs')
    ax_u.legend([f'u{i}' for i in range(nu)], loc='upper right')
    ax_u.grid(True)

    plt.xlabel('Time Step')
    plt.tight_layout()
    plt.show()

    # If xPred is not empty, plot open-loop predictions vs closed-loop for the first two states
    if xPred.size > 0 and nx >= 2: # Ensure there are at least 2 states to plot x0 vs x1
        fig_traj, ax_traj = plt.subplots(figsize=(8, 6))
        # Plot open-loop predictions
        # Collect lines for legend
        open_loop_lines = []
        for t in range(M):
            if not np.any(np.isnan(xPred[:,:,t])):
                line, = ax_traj.plot(xPred[0,:,t], xPred[1,:,t], 'r--', alpha=0.5)
                if not open_loop_lines: # Only add one instance for the legend
                    open_loop_lines.append(line)

        # Plot Closed Loop (e.g., first two states x0, x1)
        if xOpt.size > 0:
            closed_loop_line, = ax_traj.plot(xOpt[0, :], xOpt[1, :], 'bo-', label='Closed-loop trajectory')
            legend_elements = open_loop_lines + [closed_loop_line]
            ax_traj.legend(legend_elements, ['Open-loop prediction', 'Closed-loop'])


        ax_traj.set_xlabel('x')
        ax_traj.set_ylabel('y')
        ax_traj.set_title('Open-loop Predictions vs Closed-loop Trajectory (x0 vs x1)')
        ax_traj.axis('equal')
        ax_traj.grid(True)
        plt.show()

# Make sure M and N are defined. M is global from simulate_mpc cell, N is also global from il_Mog1RuMJa
# N and M are needed for the call to simulate_mpc
# M = 25 # Assuming M is available globally or will be defined correctly
# N = 3 # Assuming N is available globally or will be defined correctly

[feas, xOpt, uOpt, xPred, predErr] = simulate_mpc(P, bf, Af, x0, M)
plot_mpc_solution(feas, xOpt, uOpt, xPred, predErr)